## Group G:
1. [teammate]
2. Akhmad Fauzi Nurulhamzah

# 1. Producing the data
In this task, we will implement Apache Kafka producers to simulate real-time data streaming. Spark and parallel data processing should not be used in this section, as we are simulating sensors that often lack processing capabilities.

1.	Every 5 seconds, load 5 days of weather data from the CSV file. We refer to this as weather5s to explain the tasks; feel free to use your own variable name. You should keep a pointer in the file reading process and advance it per read. The data reading should be in chronological order.

2.	Add the current timestamp (weather_ts) to the weather5s and spread your batch out evenly for 5 seconds for each day. Since the weather data is hourly readings, each day you shall have 24 records (120 records in total for 5 days).
For example, assume you send the records at 2025-01-26 00:00:00 (ISO format: YYYY-MM-DD HH:MM:SS) -> (ts = 1737810000):
Day 1(records 1-24): ts = 1737810000
Day 2(records 25-48): ts = 1737810001
Day 3(records 49-72): ts = 1737810002
...

3.	Send your batch of weather data to a Kafka topic with an appropriate name.

# Implementation

This task implements an Apache Kafka producer to simulate real-time weather data streaming. The weather data are read chronologically in five-day batches, assigned simulated event timestamps (`weather_ts`), and published to the Kafka topic `weather5s`.

Spark is not used in this task, following the assignment requirement for simulating a lightweight sensor data source.

**Note on record count per day:** the brief's example assumes a single
site (24 hourly records/day). This dataset covers 16 sites, so each simulated day carries roughly 370-384 records (24 hours x 16 sites), consistent with the same per-day granularity the example describes, just scaled to the number of sites present in `weather.csv`.

## Configuration and Setup

The required Python libraries, Kafka broker configuration, topic name, dataset path, and streaming parameters are defined below.

In [5]:
# ==============================
# Install dependencies
# ==============================
!pip install kafka-python3 --upgrade --quiet

# ==============================
# Import statements
# ==============================
import csv
import json
import os
import time
from datetime import datetime

from kafka import KafkaProducer  # package is kafka-python3, import name stays "kafka"

# ==============================
# Kafka configuration
# ==============================
KAFKA_HOST =  "192.168.1.7"  # Kafka broker host
KAFKA_PORT = 9092
TOPIC_NAME = "weather5s"  # Task 2 subscribes to this exact name

# ==============================
# Dataset configuration
# ==============================
DATA_DIR = os.environ.get("A2B_DATA_DIR", "dataset")
WEATHER_CSV = os.path.join(DATA_DIR, "weather.csv")

# ==============================
# Batching configuration
# ==============================
DAYS_PER_BATCH = 5   # requirement 1
SECONDS_PER_DAY = 1  # requirement 2: 5 days spread over 5 seconds

# ==============================
# Sanity check
# ==============================
print("Broker :", f"{KAFKA_HOST}:{KAFKA_PORT}")
print("Topic  :", TOPIC_NAME)
print("Weather:", WEATHER_CSV, "| exists:", os.path.exists(WEATHER_CSV))

Broker : 192.168.1.7:9092
Topic  : weather5s
Weather: dataset/weather.csv | exists: True


## Load and Order Weather Data

The `load_weather_by_day` function reads the weather data and sorts the records by `(timestamp, site_id)` to ensure chronological ordering before streaming. The records are then grouped by calendar day across all available sites, allowing five consecutive historical days to be processed sequentially in each five-second batch.

The weather fields are retained as strings at the producer stage, while `weather_ts` is added as an integer timestamp. According to Task 2, the incoming weather fields are initially received as String values, except for `weather_ts`, and are converted to their required data types by the Spark Structured Streaming application.

In [6]:
# ==============================
# Reading the file into day-ordered batches
# ==============================
def load_weather_by_day(path):
    """Read weather.csv, sort chronologically, and group rows by day."""
    with open(path, newline="") as f:
        rows = list(csv.DictReader(f))

    # Sort records chronologically before streaming.
    rows.sort(key=lambda r: (r["timestamp"], int(r["site_id"])))

    days = {}
    for row in rows:
        day = row["timestamp"][:10]  # YYYY-MM-DD
        days.setdefault(day, []).append(row)

    ordered = sorted(days.items())

    print(f"Loaded {len(rows):,} rows spanning {len(ordered)} days "
          f"({ordered[0][0]} to {ordered[-1][0]})")
    print(f"Rows per day: min {min(len(d) for _, d in ordered)}, "
          f"max {max(len(d) for _, d in ordered)}")

    return ordered


weather_days = load_weather_by_day(WEATHER_CSV)


Loaded 139,773 rows spanning 365 days (2022-01-01 to 2022-12-31)
Rows per day: min 330, max 766


## Connect to Kafka

The Kafka producer connects to the configured broker and serialises each
weather record as JSON before sending it to the Kafka topic. The helper
functions handle connection and publishing exceptions so that failures can
be reported during the streaming simulation.

In [7]:
# ==============================
# Connecting to Kafka
# ==============================
def connect_kafka_producer():
    """Open the Kafka producer connection and return None if unreachable."""
    try:
        producer = KafkaProducer(
            bootstrap_servers=[f"{KAFKA_HOST}:{KAFKA_PORT}"],
            value_serializer=lambda x: json.dumps(x).encode("utf-8"),
        )
        return producer

    except Exception as ex:
        print("Exception while connecting to Kafka.")
        print(str(ex))
        return None


def publish_message(producer_instance, topic_name, data):
    """Send one JSON-serialised record to the Kafka topic."""
    try:
        producer_instance.send(topic_name, data)

    except Exception as ex:
        print("Exception while publishing message.")
        print(str(ex))


producer = connect_kafka_producer()

/tmp/ipykernel_1280053/1346604844.py:29: DeprecationWarning: value_serializer does not implement kafka.serializer.Serializer
  producer = connect_kafka_producer()


## Stream Five-Day Weather Batches

The cursor `day_pointer` represents the pointer in the file-reading process.
It advances after each five-day batch so that the historical weather data are
replayed sequentially without restarting from the beginning.

For each batch, `base_epoch` represents the simulated event time of the first
day. The following four days receive `weather_ts` values one second apart,
so five historical days are compressed into approximately five seconds of
streaming time.

The producer schedules each simulated day relative to the start of the batch
and flushes the messages after the complete five-day batch has been sent.

In [8]:
# ==============================
# The producer loop
# ==============================
def run_producer(
    days,
    producer,
    topic,
    days_per_batch=DAYS_PER_BATCH,
    max_batches=None
):
    """Publish weather data batch by batch, five days at a time."""

    day_pointer = 0
    batch_no = 0

    while day_pointer < len(days):

        if max_batches is not None and batch_no >= max_batches:
            print("Reached max_batches, stopping early.")
            break

        batch = days[day_pointer:day_pointer + days_per_batch]

        if not batch:
            break

        batch_no += 1
        batch_start = time.time()
        base_epoch = int(batch_start)
        sent = 0

        print(
            f"\n--- batch {batch_no} | "
            f"days {batch[0][0]} to {batch[-1][0]} | "
            f"base_ts {base_epoch} "
            f"({datetime.fromtimestamp(base_epoch).strftime('%Y-%m-%d %H:%M:%S')}) ---"
        )

        for offset, (day, records) in enumerate(batch):

            # Schedule each simulated day one second apart
            target_time = batch_start + offset
            remaining = target_time - time.time()

            if remaining > 0:
                time.sleep(remaining)

            weather_ts = base_epoch + offset

            for row in records:
                message = dict(row)
                message["weather_ts"] = weather_ts

                publish_message(
                    producer,
                    topic,
                    message
                )

                sent += 1

            print(
                f"  day {offset + 1} ({day}): "
                f"{len(records):>4} records, "
                f"weather_ts={weather_ts}"
            )

        # Wait until all messages in the batch are acknowledged
        producer.flush()

        day_pointer += len(batch)

        print(
            f"  batch total: {sent} records | "
            f"cursor now at day index {day_pointer}"
        )

        # Keep consecutive batches approximately five seconds apart
        remaining = batch_start + days_per_batch - time.time()

        if remaining > 0:
            time.sleep(remaining)

    print("\nProducer finished.")


if producer is None:
    print(
        "No broker connection, nothing sent. "
        "Check KAFKA_HOST and that Kafka is running."
    )

else:
    # Stream the complete weather dataset
    run_producer(
        weather_days,
        producer,
        TOPIC_NAME,
        max_batches=None
    )

    producer.flush()
    producer.close()


--- batch 1 | days 2022-01-01 to 2022-01-05 | base_ts 1788078330 (2026-08-30 18:25:30) ---
  day 1 (2022-01-01):  368 records, weather_ts=1788078330
  day 2 (2022-01-02):  376 records, weather_ts=1788078331
  day 3 (2022-01-03):  374 records, weather_ts=1788078332
  day 4 (2022-01-04):  340 records, weather_ts=1788078333
  day 5 (2022-01-05):  330 records, weather_ts=1788078334
  batch total: 1788 records | cursor now at day index 5

--- batch 2 | days 2022-01-06 to 2022-01-10 | base_ts 1788078335 (2026-08-30 18:25:35) ---
  day 1 (2022-01-06):  360 records, weather_ts=1788078335
  day 2 (2022-01-07):  374 records, weather_ts=1788078336
  day 3 (2022-01-08):  375 records, weather_ts=1788078337
  day 4 (2022-01-09):  384 records, weather_ts=1788078338
  day 5 (2022-01-10):  375 records, weather_ts=1788078339
  batch total: 1868 records | cursor now at day index 10

--- batch 3 | days 2022-01-11 to 2022-01-15 | base_ts 1788078340 (2026-08-30 18:25:40) ---
  day 1 (2022-01-11):  377 reco

  day 5 (2022-04-05):  384 records, weather_ts=1788078424
  batch total: 1915 records | cursor now at day index 95

--- batch 20 | days 2022-04-06 to 2022-04-10 | base_ts 1788078425 (2026-08-30 18:27:05) ---
  day 1 (2022-04-06):  383 records, weather_ts=1788078425
  day 2 (2022-04-07):  376 records, weather_ts=1788078426
  day 3 (2022-04-08):  384 records, weather_ts=1788078427
  day 4 (2022-04-09):  384 records, weather_ts=1788078428
  day 5 (2022-04-10):  383 records, weather_ts=1788078429
  batch total: 1910 records | cursor now at day index 100

--- batch 21 | days 2022-04-11 to 2022-04-15 | base_ts 1788078430 (2026-08-30 18:27:10) ---
  day 1 (2022-04-11):  384 records, weather_ts=1788078430
  day 2 (2022-04-12):  384 records, weather_ts=1788078431
  day 3 (2022-04-13):  383 records, weather_ts=1788078432
  day 4 (2022-04-14):  384 records, weather_ts=1788078433
  day 5 (2022-04-15):  383 records, weather_ts=1788078434
  batch total: 1918 records | cursor now at day index 105

--

  day 2 (2022-07-06):  383 records, weather_ts=1788078516
  day 3 (2022-07-07):  381 records, weather_ts=1788078517
  day 4 (2022-07-08):  384 records, weather_ts=1788078518
  day 5 (2022-07-09):  384 records, weather_ts=1788078519
  batch total: 1915 records | cursor now at day index 190

--- batch 39 | days 2022-07-10 to 2022-07-14 | base_ts 1788078520 (2026-08-30 18:28:40) ---
  day 1 (2022-07-10):  374 records, weather_ts=1788078520
  day 2 (2022-07-11):  384 records, weather_ts=1788078521
  day 3 (2022-07-12):  384 records, weather_ts=1788078522
  day 4 (2022-07-13):  382 records, weather_ts=1788078523
  day 5 (2022-07-14):  383 records, weather_ts=1788078524
  batch total: 1907 records | cursor now at day index 195

--- batch 40 | days 2022-07-15 to 2022-07-19 | base_ts 1788078525 (2026-08-30 18:28:45) ---
  day 1 (2022-07-15):  383 records, weather_ts=1788078525
  day 2 (2022-07-16):  382 records, weather_ts=1788078526
  day 3 (2022-07-17):  384 records, weather_ts=1788078527
  


--- batch 57 | days 2022-10-08 to 2022-10-12 | base_ts 1788078610 (2026-08-30 18:30:10) ---
  day 1 (2022-10-08):  384 records, weather_ts=1788078610
  day 2 (2022-10-09):  384 records, weather_ts=1788078611
  day 3 (2022-10-10):  384 records, weather_ts=1788078612
  day 4 (2022-10-11):  384 records, weather_ts=1788078613
  day 5 (2022-10-12):  384 records, weather_ts=1788078614
  batch total: 1920 records | cursor now at day index 285

--- batch 58 | days 2022-10-13 to 2022-10-17 | base_ts 1788078615 (2026-08-30 18:30:15) ---
  day 1 (2022-10-13):  384 records, weather_ts=1788078615
  day 2 (2022-10-14):  384 records, weather_ts=1788078616
  day 3 (2022-10-15):  384 records, weather_ts=1788078617
  day 4 (2022-10-16):  383 records, weather_ts=1788078618
  day 5 (2022-10-17):  384 records, weather_ts=1788078619
  batch total: 1919 records | cursor now at day index 290

--- batch 59 | days 2022-10-18 to 2022-10-22 | base_ts 1788078620 (2026-08-30 18:30:20) ---
  day 1 (2022-10-18):  38